In [44]:
import os
from bs4 import BeautifulSoup
# TimeoutError renamed as playwrightTimeout as python already has a TimeoutError
# we don't want to override the TimeoutError in Python
from playwright.async_api import async_playwright, TimeoutError as playwrightTimeout
import time
SEASONS = list(range(2020, 2027))
# Pointers to directories
DATA_DIR = "data2"
STANDING_DIR = os.path.join(DATA_DIR, "standings")
SCORES_DIR = os.path.join(DATA_DIR, "scores")

In [45]:
async def get_html(url, selector, sleep = 5, retries = 3):
    html = None
    for i in range(retries + 1):
        # prevents getting banned from the web browser from scanning too fast
        time.sleep(sleep * i)
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)
        except playwrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break
    return html           

In [46]:
async def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = await get_html(url, "#content .filter")
    
    soup = BeautifulSoup(html)
    # finds all the a tags
    # will find something link: <a href="https://google.com">Google</a>
    links = soup.find_all("a")
    # list comprehension process: turns the links into formats without the a tag such as "https://google.com"
    href = [l["href"] for l in links]
    standings_pages = [f"https://www.basketball-reference.com{l}" for l in href]
    
    for url in standings_pages:
        save_path = os.path.join(STANDING_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        # #all_schedule => the ID for the whole season table when inspected in chrome
        html = await get_html(url, "#all_schedule")
        
        if not html:
            await get_html(url, "#schedule")
        
        if not html:
            html = await get_html(url, ".table")
        
        # Opens the file with write mode
        with open(save_path, "w+") as f:
            f.write(html)

In [47]:
for season in SEASONS:
    await scrape_season(season)

2019-20 NBA Schedule | Basketball-Reference.com
2020-21 NBA Schedule | Basketball-Reference.com
2021-22 NBA Schedule | Basketball-Reference.com
2022-23 NBA Schedule | Basketball-Reference.com
2023-24 NBA Schedule | Basketball-Reference.com
2024-25 NBA Schedule | Basketball-Reference.com
2025-26 NBA Schedule | Basketball-Reference.com


In [6]:
standings_pages

NameError: name 'standings_pages' is not defined

In [48]:
standings_files = os.listdir(STANDING_DIR)

In [49]:
standings_files

['NBA_2022_games-october.html',
 'NBA_2021_games-june.html',
 'NBA_2024_games-may.html',
 'NBA_2020_games-march.html',
 'NBA_2020_games-september.html',
 'NBA_2020_games-january.html',
 'NBA_2020_games-august.html',
 'NBA_2023_games-may.html',
 'NBA_2023_games-april.html',
 'NBA_2022_games-may.html',
 'NBA_2023_games-january.html',
 'NBA_2025_games-may.html',
 'NBA_2024_games-march.html',
 'NBA_2021_games-march.html',
 'NBA_2025_games-december.html',
 'NBA_2024_games-november.html',
 'NBA_2024_games-december.html',
 'NBA_2025_games-november.html',
 'NBA_2026_games-november.html',
 'NBA_2026_games-april.html',
 'NBA_2025_games-january.html',
 'NBA_2026_games-december.html',
 'NBA_2026_games-january.html',
 'NBA_2020_games-december.html',
 'NBA_2020_games-november.html',
 'NBA_2021_games-may.html',
 'NBA_2021_games-december.html',
 'NBA_2022_games-april.html',
 'NBA_2020_games-october-2019.html',
 'NBA_2022_games-december.html',
 'NBA_2023_games-november.html',
 'NBA_2024_games-october.h

In [50]:
async def scrape_game(standings_file): 
    with open(standings_file, 'r') as f:
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get("href") for l in links]
    box_scores = [l for l in hrefs if l and "boxscore" in l and ".html" in l]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in box_scores]

    for url in box_scores:
        
        save_path = os.path.join(SCORES_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            with open(save_path, "r", encoding="utf-8") as f:
                failed_files = f.read()
            if "Rate Limited Request" not in failed_files:
                continue
            else:
                html = await get_html(url, "#content")
                if not html:
                    continue
                with open(save_path, "w+") as f:
                    f.write(html)
                time.sleep(0.5)



In [53]:
standings_files = [s for s in standings_files if ".html" in s]

for f in standings_files:
    # creates a string representing the path to a file 
    filepath = os.path.join(STANDING_DIR, f)
    
    await scrape_game(filepath)


In [33]:
standings_files

['NBA_2022_games-october.html',
 'NBA_2021_games-june.html',
 'NBA_2024_games-may.html',
 'NBA_2020_games-march.html',
 'NBA_2020_games-september.html',
 'NBA_2020_games-january.html',
 'NBA_2020_games-august.html',
 'NBA_2023_games-may.html',
 'NBA_2023_games-april.html',
 'NBA_2022_games-may.html',
 'NBA_2023_games-january.html',
 'NBA_2025_games-may.html',
 'NBA_2024_games-march.html',
 'NBA_2021_games-march.html',
 'NBA_2025_games-december.html',
 'NBA_2024_games-november.html',
 'NBA_2024_games-december.html',
 'NBA_2025_games-november.html',
 'NBA_2026_games-november.html',
 'NBA_2026_games-april.html',
 'NBA_2025_games-january.html',
 'NBA_2026_games-december.html',
 'NBA_2026_games-january.html',
 'NBA_2020_games-december.html',
 'NBA_2020_games-november.html',
 'NBA_2021_games-may.html',
 'NBA_2021_games-december.html',
 'NBA_2022_games-april.html',
 'NBA_2020_games-october-2019.html',
 'NBA_2022_games-december.html',
 'NBA_2023_games-november.html',
 'NBA_2024_games-october.h